# Optimal Steps for Fast Diffeomorphic Shape Registration: examples of the paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OWNER/REPO/blob/main/notebooks/registration_examples.ipynb)

This notebook reproduces the two registration experiments of the paper on one case each:

- **Part 1, vertebrae (surface meshes, Fig. 2)**: the C7 template
  (`data/templates_vertebrae/template_7.ply`) is registered onto the C7 vertebra segmented in the
  VerSe scan `sub-gl090`, and evaluated with the metrics of Table 1 (Chamfer, HD95,
  log-Jacobian variance).
- **Part 2, lung vessel trees (point clouds)**: the two vessel trees of the Lung250M-4B
  case 056 are registered, and evaluated with the landmark error (TRE).

It runs on **Jupyter** (from the repository, after `pip install -e ".[notebook]"`) and on
**Google Colab** (the first cell clones the repository and installs it). On Colab, select a
GPU runtime (*Runtime > Change runtime type > GPU*): the vertebra registration then takes
about a second and the lung one about 15 s. On CPU, count about 25 s for the vertebra
and 2-3 min for the lighter lung setting used there. The first run also
compiles the KeOps kernels (1-2 min).

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.exists("/content/repo"):
        !git clone --depth 1 https://github.com/OWNER/REPO.git /content/repo
    %cd /content/repo
    !pip install -q -e . plotly
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # data paths below are relative to the repository root

sys.path.insert(0, os.getcwd())
print("working directory:", os.getcwd())

## Imports

`optimal_steps` must be imported before anything that imports `pykeops`: on macOS it sets up the
compiler used by KeOps.

In [ ]:
import time

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from optimal_steps import RegistrationConfig, load_input, register, resolve_device
from optimal_steps.metrics import evaluate_registration, surface_distances

device = resolve_device("auto")

## Plotting helpers

Meshes are displayed with Plotly, which works in Jupyter, JupyterLab, VS Code and Colab
without an X server. All views use an orthographic projection; in the side-by-side plots,
both views start from the same camera.

In [ ]:
SOURCE_COLOR = "rgb(217, 38, 38)"
TARGET_COLOR = "rgb(84, 155, 235)"


def mesh_trace(mesh, color=None, name=None, opacity=1.0, intensity=None, cell_values=False,
               colorscale="Viridis", cmin=None, cmax=None, colorbar_title=None, showscale=True):
    faces = np.asarray(mesh.faces).reshape(-1, 4)[:, 1:]
    pts = np.asarray(mesh.points)
    kwargs = dict(color=color) if intensity is None else dict(
        intensity=intensity, intensitymode="cell" if cell_values else "vertex",
        colorscale=colorscale, cmin=cmin, cmax=cmax, showscale=showscale,
        colorbar=dict(title=colorbar_title, len=0.6),
    )
    return go.Mesh3d(
        x=pts[:, 0], y=pts[:, 1], z=pts[:, 2], i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
        name=name, opacity=opacity, showlegend=name is not None, flatshading=False,
        lighting=dict(ambient=0.35, diffuse=0.8, specular=0.3, roughness=0.5), **kwargs,
    )


ORTHO = dict(projection=dict(type="orthographic"))
SCENE = dict(aspectmode="data", xaxis_visible=False, yaxis_visible=False, zaxis_visible=False,
             camera=ORTHO)

def show(traces, title=None, height=550):
    fig = go.Figure(traces)
    fig.update_layout(title=title, scene=SCENE, height=height, margin=dict(l=0, r=0, t=40, b=0),
                      legend=dict(x=0.01, y=0.95))
    fig.show()


def side_by_side(left, right, titles, title=None, height=550, camera=None):
    # Two 3D views starting from the same camera. Traces with the same name share one legend entry,
    # which shows or hides them in both views.
    fig = make_subplots(rows=1, cols=2, specs=[[{"type": "scene"}] * 2],
                        horizontal_spacing=0.02, subplot_titles=titles)
    for col, traces in ((1, left), (2, right)):
        for t in traces:
            if t.name is not None:
                t.update(legendgroup=t.name, showlegend=col == 1)
            fig.add_trace(t, row=1, col=col)
    scene = dict(SCENE, camera=dict(ORTHO, **(camera or {})))
    fig.update_layout(title=title, height=height, margin=dict(l=0, r=0, t=60, b=0),
                      legend=dict(x=0.01, y=0.98), scene=scene, scene2=scene)
    fig.show()

# Part 1: vertebra registration (VerSe)

## Data

- **source**: C7 template, a refined BodyParts3D mesh (label 7),
- **target**: VerSe scan `sub-gl090`; the surface of label 7 is extracted from the
  segmentation by marching cubes, then decimated to 10k points.

In [ ]:
SOURCE = "data/templates_vertebrae/template_7.ply"
TARGET = "data/verse/dataset-01training/derivatives/sub-gl090/sub-gl090_dir-ax_seg-vert_msk.nii.gz"
LABEL = 7  # C7
MAX_POINTS = 10000

source_mesh, _ = load_input(SOURCE)
target_mesh, _ = load_input(TARGET, label=LABEL, max_points=MAX_POINTS)
print(f"source: {source_mesh.n_points} points, target: {target_mesh.n_points} points")

side_by_side([mesh_trace(source_mesh, SOURCE_COLOR)], [mesh_trace(target_mesh, TARGET_COLOR)],
             titles=("Source (C7 template)", "Target (sub-gl090, C7)"), height=450)

## Registration

Parameters of the VerSe experiments of the paper. Lengths are in mm.

In [ ]:
config = RegistrationConfig(
    sigma_init=10.0,            # kernel width at the coarsest scale
    sigma_final=4.0,            # kernel width at the finest scale
    n_scales=4,                 # coarse-to-fine levels
    outer_steps=4,              # Gauss-Newton steps per level
    lambda_reg=0.5,             # regularisation (higher = stiffer)
    use_fpfh=True,              # FPFH descriptors in the matching...
    fpfh_weight=[0.1, 0.3, 0.0, 0.0],  # ...at the two coarsest scales only
    fpfh_radius=10.0,
    normal_weight=0.05,
    use_symmetric_correspondences=True,
    trust_symmetric=0.7,        # weight of the target-to-source matches
    solver_precision_mm=0.5,
)

t0 = time.perf_counter()
result = register(
    source=source_mesh.copy(),
    target=target_mesh.copy(),
    config=config,
    rigid_align=True,
    return_history=True,        # keep the intermediate shapes for the plots below
)
elapsed = time.perf_counter() - t0

print(f"\nregistration done in {elapsed:.2f} s on {device.type.upper()}")
for step, t in result.timings.items():
    print(f"  {step:16s} {t:.2f} s")

The first run on a machine includes the compilation of the KeOps kernels. Run the cell
again to measure the registration time alone.

## Evaluation

The metrics of Table 1:
- **Chamfer distance** and **HD95** between the deformed template and the target surface,
- **variance of the log-Jacobian**: area-weighted variance of the log of the triangle area
  ratios between the deformed and the original template, which measures how unevenly the
  mesh is stretched and compressed.

In [ ]:
metrics = evaluate_registration(result.deformed_mesh, result.target_mesh, source_mesh)

aligned = result.source_mesh  # template after the rigid + anisotropic pre-alignment
chamfer_rigid = evaluate_registration(aligned, result.target_mesh, source_mesh)["chamfer_mm"]

print(f"Chamfer distance   {chamfer_rigid:.3f} mm after pre-alignment -> {metrics['chamfer_mm']:.3f} mm")
print(f"HD95               {metrics['hd95_mm']:.3f} mm")
print(f"log-Jacobian var.  {metrics['log_jacobian_var']:.3f}")

## Result

Template (red) over the target (blue, transparent), before (left) and after (right) the
diffeomorphic registration, then the distance from each vertex of the template to the target
surface. "Before" is the template after the rigid + anisotropic pre-alignment: the raw
template lies in its own coordinate frame, far from the scan.

In [ ]:
titles = (f"After pre-alignment: Chamfer {chamfer_rigid:.2f} mm",
          f"After registration: Chamfer {metrics['chamfer_mm']:.2f} mm")

side_by_side(
    [mesh_trace(aligned, SOURCE_COLOR, "template"),
     mesh_trace(result.target_mesh, TARGET_COLOR, "target", opacity=0.35)],
    [mesh_trace(result.deformed_mesh, SOURCE_COLOR, "template"),
     mesh_trace(result.target_mesh, TARGET_COLOR, "target", opacity=0.35)],
    titles=titles, title="Template and target",
)

dist_before, _ = surface_distances(aligned, result.target_mesh)
dist_after, _ = surface_distances(result.deformed_mesh, result.target_mesh)
distance_colors = dict(cmin=0, cmax=2.0, colorscale="Viridis", colorbar_title="mm")
side_by_side(
    [mesh_trace(aligned, intensity=dist_before, showscale=False, **distance_colors)],
    [mesh_trace(result.deformed_mesh, intensity=dist_after, **distance_colors)],
    titles=titles, title="Distance to the target surface (mm)",
)

## From coarse to fine

The template at the end of each scale: large kernels first align the global shape,
small ones then fit the processes. Use the slider to go through the scales.

In [ ]:
traj = result.trajectory_q  # (1 + n_scales * outer_steps, n_points, 3)
frames_idx = [0] + [(s + 1) * config.outer_steps for s in range(config.n_scales)]
sigmas = np.logspace(np.log10(config.sigma_init), np.log10(config.sigma_final), config.n_scales)
labels = ["pre-aligned"] + [f"scale {s + 1} (sigma = {sig:.1f} mm)" for s, sig in enumerate(sigmas)]

def frame_mesh(k):
    m = result.source_mesh.copy()
    m.points = traj[k]
    return m

target_trace = mesh_trace(result.target_mesh, TARGET_COLOR, "target", opacity=0.25)
fig = go.Figure(
    data=[mesh_trace(frame_mesh(0), SOURCE_COLOR, "template"), target_trace],
    frames=[go.Frame(data=[mesh_trace(frame_mesh(k), SOURCE_COLOR, "template"), target_trace], name=lab)
            for k, lab in zip(frames_idx, labels)],
)
fig.update_layout(
    scene=SCENE, height=600, margin=dict(l=0, r=0, t=40, b=0), title="Coarse-to-fine registration",
    sliders=[dict(steps=[dict(method="animate", label=lab,
                              args=[[lab], dict(mode="immediate", frame=dict(duration=0, redraw=True))])
                         for lab in labels],
                  currentvalue=dict(prefix=""), pad=dict(t=10))],
)
fig.show()

## Mesh quality

Log of the area ratio of every triangle between the deformed and the original template.
The deformation is a diffeomorphism, so the mesh never folds; its local expansions and
compressions stay moderate, which keeps the triangulation usable for downstream analyses.

In [ ]:
area_src = source_mesh.compute_cell_sizes(length=False, volume=False)["Area"]
area_def = result.deformed_mesh.compute_cell_sizes(length=False, volume=False)["Area"]
log_ratio = np.log((area_def + 1e-12) / (area_src + 1e-12))
log_ratio -= np.average(log_ratio, weights=area_src)  # remove the global scaling

show([
    mesh_trace(result.deformed_mesh, intensity=log_ratio, cell_values=True,
               colorscale="RdBu_r", cmin=-1, cmax=1, colorbar_title="log area ratio"),
], title="Local expansion (red) and compression (blue)")

## Save the vertebra meshes

In [ ]:
out_dir = "results/notebook_verse"
os.makedirs(out_dir, exist_ok=True)
source_mesh.save(f"{out_dir}/source_template.ply")                 # template, in its own frame
result.source_mesh.save(f"{out_dir}/source_template_prealigned.ply")  # after the pre-alignment
result.deformed_mesh.save(f"{out_dir}/deformed_template.ply")
result.target_mesh.save(f"{out_dir}/target.ply")
print("saved in", out_dir)

# On Colab, to download a file: from google.colab import files; files.download(path)

# Part 2: lung vessel trees (Lung250M-4B)

Each Lung250M-4B case provides the vessel trees of the same patient at inhale and exhale as
point clouds, with, for every point, an **artery / vein label** and the **vessel radius**,
plus 100 manually placed landmark pairs (99 for this case) to measure the accuracy.
`data/lungs/lung250m4b_case_056.npz` contains case 056 at two resolutions: ~30k points
(the setting of the paper) and 8k points.

Point clouds have no normals, so the matching score combines the position, the vessel
radius, the vessel direction (local PCA) and the artery/vein label (see `register_lungs.py`);
the deformation model is the same as for the vertebrae. The whole pipeline, including the pre-alignment
(centring, anisotropic scaling, ICP), is in `register_lungs.register_clouds`.

On a GPU we use the paper's setting. On CPU, where it would take hours, we use the 8k-point
clouds and fewer, larger steps: the result is close but not identical to the paper's.

In [ ]:
import pyvista as pv
import torch
import register_lungs as rl

lung = np.load("data/lungs/lung250m4b_case_056.npz")

ON_GPU = device.type == "cuda"
RES = "30k" if ON_GPU else "8k"
lung_config = RegistrationConfig(**rl.LUNG_CONFIG) if ON_GPU else RegistrationConfig(
    **{**rl.LUNG_CONFIG, "outer_steps": 4, "euler_precision_step_mm": 0.5})

src_pts, tgt_pts = lung[f"source_points_{RES}"], lung[f"target_points_{RES}"]
src_av, tgt_av = lung[f"source_artery_vein_{RES}"], lung[f"target_artery_vein_{RES}"]
src_r, tgt_r = lung[f"source_radius_{RES}"], lung[f"target_radius_{RES}"]
lm_src, lm_tgt = lung["landmarks_source"], lung["landmarks_target"]

print(f"resolution {RES}: source {len(src_pts)} points, target {len(tgt_pts)} points, "
      f"{len(lm_src)} landmarks")

In [ ]:
AV_COLORS = [[0, "rgb(217, 38, 38)"], [1, "rgb(40, 90, 200)"]]  # arteries / veins


def cloud_trace(points, color=None, values=None, name=None, size=1.5, opacity=1.0,
                colorscale=AV_COLORS, showscale=False):
    marker = dict(size=size, opacity=opacity)
    if values is not None:
        marker.update(color=values, colorscale=colorscale, showscale=showscale)
    else:
        marker.update(color=color)
    return go.Scatter3d(x=points[:, 0], y=points[:, 1], z=points[:, 2], mode="markers",
                        marker=marker, name=name, showlegend=name is not None)


# Frontal view: x runs left-right, y from the apex to the base.
LUNG_CAMERA = dict(eye=dict(x=0, y=0, z=-1.6), up=dict(x=0, y=-1, z=0))

side_by_side([cloud_trace(src_pts, values=src_av)], [cloud_trace(tgt_pts, values=tgt_av)],
             titles=("Source tree (arteries red, veins blue)", "Target tree"),
             height=500, camera=LUNG_CAMERA)

## Registration

In [ ]:
t0 = time.perf_counter()
displacement, lung_traj = rl.register_clouds(
    torch.from_numpy(src_pts), torch.from_numpy(tgt_pts),
    src_av[:, None], src_r[:, None], tgt_av[:, None], tgt_r[:, None],
    lung_config,
    return_history=True,  # keep the intermediate shapes for the slider below
)
displacement = displacement.numpy()
lung_elapsed = time.perf_counter() - t0
deformed_pts = src_pts + displacement
print(f"\nregistration done in {lung_elapsed:.1f} s on {device.type.upper()}")

## From coarse to fine

The source tree before any alignment, after the pre-alignment, then at the end of each
scale: the large kernels first register the whole lungs, the small ones then align the
individual vessels. Use the slider to go through the steps.

In [ ]:
lung_frames_idx = [0] + [(s + 1) * lung_config.outer_steps for s in range(lung_config.n_scales)]
lung_sigmas = np.logspace(np.log10(lung_config.sigma_init), np.log10(lung_config.sigma_final),
                          lung_config.n_scales)
lung_states = [src_pts] + [lung_traj[k] for k in lung_frames_idx]
lung_labels = ["initial", "pre-aligned"] + [f"scale {s + 1} (sigma = {sig:.1f} mm)"
                                            for s, sig in enumerate(lung_sigmas)]

lung_target_trace = cloud_trace(tgt_pts, TARGET_COLOR, name="target", opacity=0.4)
fig = go.Figure(
    data=[cloud_trace(lung_states[0], SOURCE_COLOR, name="source"), lung_target_trace],
    frames=[go.Frame(data=[cloud_trace(pts, SOURCE_COLOR, name="source"), lung_target_trace], name=lab)
            for pts, lab in zip(lung_states, lung_labels)],
)
fig.update_layout(
    scene=dict(SCENE, camera=dict(ORTHO, **LUNG_CAMERA)), height=650, margin=dict(l=0, r=0, t=40, b=0),
    title="Coarse-to-fine registration of the vessel trees", legend=dict(x=0.01, y=0.95),
    sliders=[dict(steps=[dict(method="animate", label=lab,
                              args=[[lab], dict(mode="immediate", frame=dict(duration=0, redraw=True))])
                         for lab in lung_labels],
                  currentvalue=dict(prefix=""), pad=dict(t=10))],
)
fig.show()

## Landmark error

The displacement is interpolated at the source landmarks (Gaussian weights on the 15
nearest points), and the warped landmarks are compared with the target ones.

In [ ]:
lms = np.hstack([lm_src, lm_tgt])
tre_before, tre_after = rl.landmark_errors(src_pts, displacement, lms)
print(f"TRE before registration  {tre_before.mean():.2f} ± {tre_before.std():.2f} mm")
print(f"TRE after registration   {tre_after.mean():.2f} ± {tre_after.std():.2f} mm")
print("(paper: 3.07 mm on average over the 27 test cases, skeletonized clouds on GPU)")

fig = go.Figure([
    go.Box(y=tre_before, name="before", boxpoints="all", jitter=0.4, marker_color="gray"),
    go.Box(y=tre_after, name="after", boxpoints="all", jitter=0.4, marker_color=TARGET_COLOR),
])
fig.update_layout(title="Target registration error on the landmarks", yaxis_title="mm",
                  height=400, showlegend=False)
fig.show()

## Result

Source tree (red) and target tree (blue), before (left) and after (right) registration.
Black: target landmarks; orange: source landmarks, as given on the left and warped by
the registration on the right. Each segment joins a pair of landmarks: its length is the
error.

In [ ]:
warped_lm = lm_src + rl.interpolate_displacements(src_pts, displacement, lm_src)


def landmark_segments(a, b):
    seg = np.full((3 * len(a), 3), np.nan)  # NaN rows break the line between pairs
    seg[0::3], seg[1::3] = a, b
    return seg


def registration_traces(source, source_lm):
    seg = landmark_segments(source_lm, lm_tgt)
    return [
        cloud_trace(source, SOURCE_COLOR, name="source tree", opacity=0.6),
        cloud_trace(tgt_pts, TARGET_COLOR, name="target tree", opacity=0.6),
        cloud_trace(lm_tgt, "black", name="target landmarks", size=4),
        cloud_trace(source_lm, "orange", name="source landmarks", size=4),
        go.Scatter3d(x=seg[:, 0], y=seg[:, 1], z=seg[:, 2], mode="lines",
                     line=dict(color="black", width=3), name="landmark error"),
    ]


side_by_side(
    registration_traces(src_pts, lm_src),
    registration_traces(deformed_pts, warped_lm),
    titles=(f"Before registration: TRE {tre_before.mean():.2f} mm",
            f"After registration: TRE {tre_after.mean():.2f} mm"),
    title="Source and target vessel trees", height=650, camera=LUNG_CAMERA,
)

In [ ]:
out_dir = "results/notebook_lungs"
os.makedirs(out_dir, exist_ok=True)
# Point clouds as .ply, with the artery/vein label and the vessel radius of every point.
for name, pts, av, radius in (("source", src_pts, src_av, src_r),
                              ("deformed", deformed_pts, src_av, src_r),
                              ("target", tgt_pts, tgt_av, tgt_r)):
    cloud = pv.PolyData(np.asarray(pts, dtype=np.float32))
    cloud.point_data["artery_vein"], cloud.point_data["radius"] = av, radius
    cloud.save(f"{out_dir}/case_056_{name}_{RES}.ply")
np.save(f"{out_dir}/case_056_displacement_{RES}.npy", displacement)
print("saved in", out_dir)

## Going further

- `register_batch.py`: the vertebra registration for every row of a CSV manifest, with a
  summary of the metrics (`python register_batch.py --manifest examples/manifest_example.csv`),
- `register_vertebrae.py`: all the VerSe vertebrae, with the metrics of Table 1 and the atlases
  (mean shape and modes of variation),
- `register_lungs.py`: all the Lung250M-4B cases and their landmark errors.